# 1 · Train the RL traffic-light controllers (Colab)

This notebook trains DQN, Double DQN, PPO and A2C on the RESCO scenarios with `run_all.py`.
Results (models, logs) are saved in **your Google Drive**, so they are kept if Colab disconnects.

**Before you start**

* **Runtime → Change runtime type → CPU** is the right choice. A GPU does **not** make this faster:
  most of the time is spent in the SUMO traffic simulator, which only uses the CPU, and the
  networks are tiny (64×64), so moving each decision to a GPU costs more than it saves.
  What speeds things up is **more CPU cores**. The setup cell prints how many you have.
  Free Colab usually gives 2; your PC has 8.
* **Best use of Colab: run it at the same time as your PC.** For example, the PC trains
  `cologne1` + `ingolstadt1` and Colab trains `cologne3` (set `SCENARIOS` below).
  To work from one results folder, put the PC's `results/` into the same Drive folder
  (e.g. with Google Drive for desktop).
* Keep this browser tab open. Free sessions end after some hours. **If it disconnects, run all
  cells again:** finished runs are skipped. A run that was interrupted starts again from episode 1,
  so at most one run per core is lost.

## 1. Settings: what this session trains

In [ ]:
SCENARIOS = ["cologne3"]   # e.g. ["cologne3"] if the PC does the other two
ALGOS     = ["dqn", "ddqn", "ppo", "a2c"]
SEEDS     = [0, 1]
NET       = [64, 64]          # hidden layers; [16, 16] or [256, 256] for the network-size ablation
TRAIN_SET = "variants"        # "real" = ablation: train on the real hour only
EPISODES  = 200
WORKERS   = None              # None = one run per CPU core

RESULTS = "/content/drive/MyDrive/icarc_results"
REPO    = "https://github.com/Konara98/icarc_dataset_scripts.git"

## 2. Setup

In [ ]:
# Setup: Google Drive, code, packages. Takes about 1-2 minutes.
import os, subprocess, sys, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(RESULTS, exist_ok=True)
os.environ["TSC_RESULTS"] = RESULTS          # every script saves into this Drive folder

CODE = "/content/icarc_dataset_scripts"
if os.path.exists(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
os.chdir(CODE)

# Same versions as the local PC, so results from both machines are comparable.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eclipse-sumo==1.27.1", "libsumo==1.27.1",
                "sumo-rl==1.4.5", "stable-baselines3==2.9.0"], check=True)

CPUS = os.cpu_count()
print(f"code: {CODE} ({subprocess.run(['git', 'log', '-1', '--format=%h %cd'], capture_output=True, text=True).stdout.strip()})")
print(f"results: {RESULTS}")
print(f"CPU cores: {CPUS}")

In [ ]:
# Quick check: one simulated hour of fixed-time control on cologne1 (should take a few seconds)
check = subprocess.run([sys.executable, "-c", """
import time, common as C
t = time.time()
m = C.run_episode('cologne1', C.real_hour_file('cologne1'), 'fixed', sumo_seed=1000)
import os
print(f"libsumo: {os.environ.get('LIBSUMO_AS_TRACI') is not None}, one hour simulated in {time.time()-t:.1f} s,",
      f"avg total wait {m['avg_total_wait']:.1f} s, finished {m['finished']}/{m['vehicles']}")
"""], capture_output=True, text=True)
print(check.stdout or check.stderr[-3000:])

## 3. Train

This cell runs until every run is finished. It refreshes a progress table every 2 minutes.
On your PC one run took about 45 minutes with 6 runs in parallel; cologne3 runs take longer.

In [ ]:
import csv, json, glob
from IPython.display import clear_output

workers = WORKERS or CPUS
tag = f"{{algo}}_{'x'.join(map(str, NET))}_{TRAIN_SET}"


def progress():
    rows = []
    for s in SCENARIOS:
        for algo in ALGOS:
            for seed in SEEDS:
                d = os.path.join(RESULTS, s, tag.format(algo=algo), f"seed{seed}")
                ep, best, state = 0, "", "waiting"
                try:
                    with open(os.path.join(d, "train_log.csv")) as fh:
                        ep = sum(1 for _ in fh) - 1
                    state = "running"
                    with open(os.path.join(d, "val_log.csv")) as fh:
                        vals = [float(r["val_avg_total_wait"]) for r in csv.DictReader(fh)]
                    best = f"{min(vals):.1f}" if vals else ""
                    with open(os.path.join(d, "config.json")) as fh:
                        c = json.load(fh)
                    if "finished" in c and c["episodes"] == EPISODES:
                        state = "done"
                except (OSError, ValueError):
                    pass
                rows.append((s, tag.format(algo=algo), seed, state, ep, best))
    return rows


cmd = [sys.executable, "run_all.py", "--scenarios", *SCENARIOS, "--algos", *ALGOS,
       "--seeds", *map(str, SEEDS), "--net", *map(str, NET), "--train-set", TRAIN_SET,
       "--episodes", str(EPISODES), "--workers", str(workers)]
log_path = os.path.join(RESULTS, "run_all.log")
proc = subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
start = time.time()

while True:
    finished = proc.poll() is not None
    clear_output(wait=True)
    rows = progress()
    done = sum(r[3] == "done" for r in rows)
    print(f"{done}/{len(rows)} runs done · {workers} at a time · {(time.time() - start) / 3600:.1f} h elapsed\n")
    print(f"{'scenario':12} {'configuration':24} {'seed':>4}  {'state':8} {'episode':>9}  best validation wait (s)")
    for s, t, seed, state, ep, best in rows:
        print(f"{s:12} {t:24} {seed:4d}  {state:8} {ep:4d}/{EPISODES:<4d}  {best}")
    print("\nrun_all.py log:\n" + "".join(open(log_path).readlines()[-6:]))
    if finished:
        break
    time.sleep(120)

print("Training finished." if proc.returncode == 0 else f"run_all.py exited with code {proc.returncode}")

## 4. Learning curves

The policy's score on the 5 **validation** files during training: mean over seeds (line) and
spread between seeds (band). The dashed line is fixed-time control on the same files. It appears
once notebook 2 has tuned the baselines; until then the curves show without it. Lower is better.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

COLORS = {"dqn": "#2a78d6", "ddqn": "#eb6834", "ppo": "#1baf7a", "a2c": "#eda100"}
NAMES = {"dqn": "DQN", "ddqn": "Double DQN", "ppo": "PPO", "a2c": "A2C"}

fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(5.2 * len(SCENARIOS), 3.8), squeeze=False)
for ax, s in zip(axes[0], SCENARIOS):
    top = 0
    for algo in ALGOS:
        frames = [pd.read_csv(f) for f in glob.glob(os.path.join(RESULTS, s, tag.format(algo=algo), "seed*", "val_log.csv"))]
        frames = [f for f in frames if len(f)]
        if not frames:
            continue
        df = pd.concat(frames).groupby("episode")["val_avg_total_wait"].agg(["mean", "std"]).fillna(0)
        ax.fill_between(df.index, df["mean"] - df["std"], df["mean"] + df["std"], color=COLORS[algo], alpha=0.15, lw=0)
        ax.plot(df.index, df["mean"], color=COLORS[algo], lw=2, label=f"{NAMES[algo]} ({len(frames)} seeds)")
        top = max(top, df["mean"].iloc[len(df) // 2:].max())
    p = os.path.join(RESULTS, s, "actuated_params.json")
    if os.path.exists(p):
        fixed = json.load(open(p))["fixed_val_score"]
        ax.axhline(fixed, color="#52514e", lw=1.5, ls="--", label="fixed-time")
        top = max(top, fixed)
    ax.set_ylim(0, top * 2 if top else None)   # early, untrained episodes are off the chart
    ax.set_title(s, loc="left", fontsize=11)
    ax.set_xlabel("training episode")
    ax.set_ylabel("validation total wait per vehicle (s)")
    ax.grid(axis="y", color="#e6e5e0", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

Next: open **02_evaluate.ipynb** to test every controller on the test files.